# E-commerce Performance Analytics — Sales, Funnel & Acquisition

This notebook walks through the analysis end to end: data quality checks,
sales performance, the conversion funnel, and acquisition channel
performance, finishing with the findings that feed `docs/business_recommendations.md`.

Dataset: synthetic e-commerce data for a fictional 4x4/auto parts retailer
("TrailHub Parts"). See `docs/methodology.md` for how the data was built
and its limitations.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

products = pd.read_csv("../data/products.csv")
customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
website = pd.read_csv("../data/website_daily.csv", parse_dates=["date"])

print("products:", products.shape)
print("customers:", customers.shape)
print("orders:", orders.shape)
print("website_daily:", website.shape)


products: (150, 8)
customers: (320, 6)
orders: (650, 7)
website_daily: (365, 8)


## 1. Data quality

Check missing values, duplicates and referential integrity before trusting any downstream number.

In [2]:
for name, df in [("products", products), ("customers", customers), ("orders", orders), ("website", website)]:
    n_missing = df.isna().sum().sum()
    n_dupes = df.duplicated().sum()
    print(f"{name}: {len(df)} rows, {n_missing} missing values, {n_dupes} duplicate rows")

orphan_products = orders.loc[~orders["product_id"].isin(products["product_id"])]
orphan_customers = orders.loc[~orders["customer_id"].isin(customers["customer_id"])]
print("\norders with unknown product_id:", len(orphan_products))
print("orders with unknown customer_id:", len(orphan_customers))


products: 150 rows, 0 missing values, 0 duplicate rows
customers: 320 rows, 0 missing values, 0 duplicate rows
orders: 650 rows, 0 missing values, 0 duplicate rows
website: 365 rows, 0 missing values, 0 duplicate rows

orders with unknown product_id: 0
orders with unknown customer_id: 0


No missing values or duplicates, and every order links to a real product and customer. The dataset is clean because it's synthetic — a real extract from production would need more work here, which is exactly what `python/data_cleaning.py` is set up to handle.

## 2. Sales performance

In [3]:
total_orders = len(orders)
total_revenue = orders["revenue"].sum()
aov = total_revenue / total_orders
merged = orders.merge(products[["product_id","cost","category"]], on="product_id")
margin = (merged["revenue"] - merged["quantity"]*merged["cost"]).sum()

print(f"Orders (sample extract): {total_orders:,}")
print(f"Revenue (sample extract): EUR {total_revenue:,.0f}")
print(f"Average order value: EUR {aov:,.2f}")
print(f"Gross margin (sample extract): EUR {margin:,.0f} ({100*margin/total_revenue:.1f}%)")


Orders (sample extract): 650
Revenue (sample extract): EUR 135,223
Average order value: EUR 208.04
Gross margin (sample extract): EUR 58,907 (43.6%)


In [4]:
by_category = merged.groupby("category").agg(
    orders=("order_id","nunique"),
    revenue=("revenue","sum"),
).sort_values("revenue", ascending=False)
by_category["revenue_share_pct"] = (100*by_category["revenue"]/by_category["revenue"].sum()).round(1)
by_category


In [5]:
product_revenue = merged.groupby("product_id")["revenue"].sum().sort_values(ascending=False)
top_decile_share = product_revenue.head(int(len(product_revenue)*0.1)).sum() / product_revenue.sum()
print(f"Top 10% of products by revenue generate {100*top_decile_share:.1f}% of total revenue in this sample.")


Top 10% of products by revenue generate 30.6% of total revenue in this sample.


## 3. Conversion funnel

In [6]:
funnel_totals = website[["sessions","product_views","add_to_cart","checkout","purchases","revenue"]].sum()
cart_rate = 100*funnel_totals["add_to_cart"]/funnel_totals["sessions"]
checkout_completion = 100*funnel_totals["checkout"]/funnel_totals["add_to_cart"]
purchase_completion = 100*funnel_totals["purchases"]/funnel_totals["checkout"]
overall_cvr = 100*funnel_totals["purchases"]/funnel_totals["sessions"]
rev_per_session = funnel_totals["revenue"]/funnel_totals["sessions"]

print(f"Sessions -> Add to cart:  {cart_rate:.2f}%")
print(f"Add to cart -> Checkout:  {checkout_completion:.2f}%")
print(f"Checkout -> Purchase:     {purchase_completion:.2f}%")
print(f"Overall conversion rate:  {overall_cvr:.2f}%")
print(f"Revenue per session:      EUR {rev_per_session:.2f}")


Sessions -> Add to cart:  9.90%
Add to cart -> Checkout:  60.75%
Checkout -> Purchase:     67.68%
Overall conversion rate:  4.07%
Revenue per session:      EUR 5.62


In [7]:
website["month"] = website["date"].dt.to_period("M")
monthly = website.groupby("month")[["sessions","add_to_cart","checkout","purchases"]].sum()
monthly["conversion_rate_pct"] = (100*monthly["purchases"]/monthly["sessions"]).round(2)
monthly.tail(6)


## 4. Acquisition channels

In [8]:
by_channel = orders.groupby("acquisition_channel").agg(
    orders=("order_id","count"),
    revenue=("revenue","sum"),
).sort_values("revenue", ascending=False)
by_channel["revenue_share_pct"] = (100*by_channel["revenue"]/by_channel["revenue"].sum()).round(1)
by_channel["avg_order_value"] = (by_channel["revenue"]/by_channel["orders"]).round(2)
by_channel


## 5. Key findings

1. Organic search is the largest acquisition channel by order volume and revenue in this sample, ahead of paid search and direct — consistent with a catalogue-driven business where product and category pages carry a lot of the discovery weight.
2. The biggest single drop-off in the funnel is between session and add-to-cart, not at checkout. That points to product findability and product-page relevance as the first place to fix, before touching the checkout flow.
3. Revenue is concentrated: roughly the top 10% of SKUs by revenue account for a disproportionate share of total revenue, which is the classic case for prioritising catalogue and SEO effort on a shortlist of products rather than spreading it evenly.
4. Checkout completion (checkout started -> purchase) is meaningfully higher than the add-to-cart -> checkout-started step, meaning people who commit to checking out mostly finish. The leak is earlier, in getting people from cart to checkout at all.

Full writeup with business impact and recommendations: `docs/business_recommendations.md`.